RANDOM FOREST REGRESSOR MODEL

Michael Owens 

In [ ]:
# Importing functions from various libraries (Extra tools are brought in as well)
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

In [ ]:
# Bringing in the Data

# Read data
df = pd.read_csv('CleanPreprocessedExoplanet2.csv') # This is the output from the preprocessing - file name "Preprocessing.ipynb"
df = df.drop('Unnamed: 0',axis=1) # Remove relic of old data table 
print(df.shape)
df.head(3)

(1788, 50)


,release_month,sy_snum,sy_pnum,cb_flag,ttv_flag,pl_nnotes,elon,elat,glon,glat,...,st_logg,st_mass,soltype_Kepler Project Candidate (q1_q17_dr25_koi),soltype_Published Confirmed,pl_tsystemref_BJD,pl_tsystemref_BJD-TDB,pl_tsystemref_BJD-UTC,pl_tsystemref_HJD,pl_tsystemref_JD,log radius
0,1,1,2,0,0,0,182.62760,3.28517,282.19917,63.35883,...,4.435,0.958,False,True,True,False,False,False,False,0.194514
1,11,1,1,0,0,0,173.16246,3.72695,261.10589,63.11979,...,4.595,0.802,False,True,True,False,False,False,False,0.017033
2,3,1,2,0,0,0,336.33924,1.30362,56.17023,-51.54478,...,4.596,0.832,False,True,False,False,False,False,True,0.320977


Grid Search to Determine the Number of Estimators and Tree Depth

In [3]:
#Split data & define features/targets
(df_train,df_test) = train_test_split(df,train_size=0.8,
                                      test_size=0.2,
                                      random_state=0)
X_train = df_train.drop('log radius',axis=1)
X_test  = df_test.drop('log radius',axis=1)
y_train   = df_train['log radius']
y_test    = df_test['log radius']

Initial grid search to find optimal hyperparameters

In [ ]:
grid = {'max_depth' : np.arange(1,40,3),'n_estimators':np.arange(1,5000,500)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1) # Grid Searching for hyperparameters
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(28), 'n_estimators': np.int64(1501)}
    Optimal Valid R2 = 0.3840864178335779


Implementing a Heap Map to Assist Grid Search

In [6]:
Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

In [ ]:
fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show() # Heatmap to help show areas of further searching

Refined Grid Search

In [ ]:
# Refined search to look more percisely in the region of the optimal found in the first round of searching

grid = {'max_depth' : np.arange(26,30,1),'n_estimators':np.arange(1000,2000,50)}
rfr = RandomForestRegressor(max_features = 1/3)
rfrCV = GridSearchCV(rfr,param_grid= grid,n_jobs=-1,verbose=2)
rfrCV.fit(X_train,y_train)
print('Random Forest Regressor:')
print('    Optimal Parameters:', rfrCV.best_params_ )
print('    Optimal Valid R2 =', rfrCV.best_score_ )

Scores_mean = rfrCV.cv_results_['mean_test_score']
Scores_mean = Scores_mean.reshape(len(grid['max_depth']),len(grid['n_estimators']))

fig = px.imshow(Scores_mean,labels=dict(x='Number of Estimators',y='Number of Tree Depth'),y=grid['max_depth'],x=grid['n_estimators'],aspect='auto')
fig.show() # Producing a Heatmap to help evaluate wheter there is a need to do another layer of searching to develop hyperparameters

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Random Forest Regressor:
    Optimal Parameters: {'max_depth': np.int64(28), 'n_estimators': np.int64(1600)}
    Optimal Valid R2 = 0.38370889456582047


Evaluating the Model

In [ ]:
# Printing the R2 Values on the training and testing data sets
print(f" train R2 {rfrCV.score(X_train,y_train):.3f}")
print(f" test R2 {rfrCV.score(X_test,y_test):.3f}")

 train R2 0.919
 test R2 0.478


Providing the MSE for the training and testing sets

In [11]:
rf = RandomForestRegressor(max_depth = 20,n_estimators=485,max_features = 1/3,oob_score=True)
rf.fit(X_train,y_train)

print(f'out-of-bag R2 = {rf.oob_score_:.3f}')
print()
print(f'training R2 {rf.score(X_train,y_train)}')
print(f'testing R2: {rf.score(X_test,y_test)}')

from sklearn.metrics import mean_squared_error
y_pred_test = rf.predict(X_test)
y_pred_train = rf.predict(X_train)

print(f'MSE Test: {mean_squared_error(y_test,y_pred_test)}')
print(f'MSE Train: {mean_squared_error(y_train,y_pred_train)}')


out-of-bag R2 = 0.397

training R2 0.9147751334036791
testing R2: 0.47529384595418744
MSE Test: 0.037578807870141166
MSE Train: 0.00658329407505394


In [ ]:
# Provides the value of relative uncertainity for the model and the max/min uncertainity % in predictions 
rel_unc = abs((10**y_pred_test)-(10**y_test))/(10**y_test)

print(max(rel_unc)) # Maximum
print(min(rel_unc)) # Minimum
rel_unc.mean()      # Overall

1.7680695396145059
0.001305315124072272


np.float64(0.3317077578821904)